# prep: `company_jobdescription.csv` — `job_types` 빈 값 행 제거

- **목적:** JD 데이터에서 `job_types` 컬럼(엑셀 기준 D열)이 비어 있는 행을 모두 제거한 정제 버전 생성.
- **이유:** 빈 `job_types`는 채용공고가 아니거나 크롤링 누락된 레코드로 추정 — 매칭 파이프라인 노이즈로 작용.
- **입력:** `/raw/data/company_jobdescription.csv` (235,850 rows × 52 cols)
- **출력:** `/raw/data/company_jobdescription_preprocessing.csv`
- **관련 위키:** raw-company-jobdescription-EDA
- **작성일:** 2026-05-18

### 빈 값 판정 규칙
다음을 모두 '빈 값'으로 간주하여 제거:
- `NaN` / `None`
- 빈 문자열 `''`, 공백 `' '`
- 빈 리스트 문자열 `'[]'`, `'[ ]'`
- 문자열 `'nan'`, `'null'`, `'None'` (대소문자 무관)

In [2]:
import pandas as pd
from pathlib import Path

RAW_DATA = Path('raw/data')
SRC = RAW_DATA / 'company_jobdescription.csv'
DST = RAW_DATA / 'company_jobdescription_preprocessing.csv'

assert SRC.exists(), f'원본 없음: {SRC}'
print('SRC:', SRC)
print('DST:', DST)

SRC: raw/data/company_jobdescription.csv
DST: raw/data/company_jobdescription_preprocessing.csv


In [3]:
df = pd.read_csv(SRC, encoding='utf-8-sig', low_memory=False)
print(f'원본: {len(df):,} rows × {len(df.columns)} cols')
print(f'\n컬럼 D (index 3) = {df.columns[3]!r}')
assert df.columns[3] == 'job_types', f'예상 컬럼 불일치: {df.columns[3]}'
df['job_types'].head(10)

원본: 235,850 rows × 8 cols

컬럼 D (index 3) = 'job_types'


0    EXPERIENCED, NEW, CONTRACT
1              NEW, EXPERIENCED
2                           NaN
3                   EXPERIENCED
4                        INTERN
5              NEW, EXPERIENCED
6                           NaN
7                           NaN
8                           NaN
9                           NaN
Name: job_types, dtype: str

In [4]:
def is_empty(v) -> bool:
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == '' or s == '[]' or s == '[ ]':
        return True
    if s.lower() in {'nan', 'null', 'none'}:
        return True
    return False

mask_empty = df['job_types'].apply(is_empty)
n_empty = int(mask_empty.sum())
n_keep = len(df) - n_empty
print(f'job_types 빈 값 행:   {n_empty:>8,}  ({n_empty/len(df)*100:5.2f}%)')
print(f'job_types 유효 행:    {n_keep:>8,}  ({n_keep/len(df)*100:5.2f}%)')
print(f'합계:                 {len(df):>8,}')

job_types 빈 값 행:    122,619  (51.99%)
job_types 유효 행:     113,231  (48.01%)
합계:                  235,850


In [5]:
df_clean = df[~mask_empty].reset_index(drop=True)
print(f'정제 결과: {len(df_clean):,} rows × {len(df_clean.columns)} cols')

df_clean.to_csv(DST, index=False, encoding='utf-8-sig')
print(f'\nsaved: {DST}')
print(f'파일 크기: {DST.stat().st_size / (1024*1024):.1f} MB')

정제 결과: 113,231 rows × 8 cols

saved: raw/data/company_jobdescription_preprocessing.csv
파일 크기: 131.7 MB


In [6]:
print('=== 제거 전후 비교 ===')
print(f'before : {len(df):>8,} rows')
print(f'after  : {len(df_clean):>8,} rows')
print(f'removed: {len(df) - len(df_clean):>8,} rows  ({(len(df)-len(df_clean))/len(df)*100:.2f}%)')

print('\n=== 정제본 job_types 상위 10개 ===')
print(df_clean['job_types'].value_counts().head(10))

=== 제거 전후 비교 ===
before :  235,850 rows
after  :  113,231 rows
removed:  122,619 rows  (51.99%)

=== 정제본 job_types 상위 10개 ===
job_types
NEW                           33757
INTERN                        32767
NEW, EXPERIENCED              19724
CONTRACT                      13239
EXPERIENCED                    5292
EXPERIENCED, NEW               1608
NEW, EXPERIENCED, CONTRACT     1420
NEW, CONTRACT                  1106
INTERN, NEW                     643
INTERN, EXPERIENCED             429
Name: count, dtype: int64


## 2차 필터 — `detail_text_clean` 빈 값 + 짧은 행 제거 (2026-05-28 추가)

1차(`job_types` 빈 값 제거) 통과한 113,231행 중에서 추가로 다음 조건 행을 제거:
- `detail_text_clean` 이 빈 값 (NaN/`''`/`'[]'`/`'nan'/'null'/'None'`)
- `detail_text_clean` 의 **문자 길이 ≤ 120자** — 매칭 신호 부족

> 종전 *"2줄 이하"* 룰은 폐기 — `detail_text_clean`은 cleaning 과정에서 줄바꿈이 모두 제거된 한 줄 텍스트이므로 줄 카운트가 무의미. 사용자 결정(2026-05-28)으로 **120자 임계값** 채택 — `gemini_profile_faiss_matching.ipynb`의 `jd_text_len >= 120` 관례와 정합.

**출력 덮어쓰기:** `/raw/data/company_jobdescription_preprocessing.csv`

In [7]:
assert 'detail_text_clean' in df_clean.columns, f'detail_text_clean 컬럼 없음: {list(df_clean.columns)}'

MIN_LEN = 120

mask_null = df_clean['detail_text_clean'].apply(is_empty)
text_len = df_clean['detail_text_clean'].fillna('').astype(str).str.len()
mask_short = text_len < MIN_LEN
mask_drop = mask_null | mask_short

n_null = int(mask_null.sum())
n_short_only = int((mask_short & ~mask_null).sum())
n_drop = int(mask_drop.sum())
n_keep = len(df_clean) - n_drop

print(f'detail_text_clean 빈 값:              {n_null:>8,}  ({n_null/len(df_clean)*100:5.2f}%)')
print(f'detail_text_clean <{MIN_LEN}자 (빈값 제외): {n_short_only:>8,}  ({n_short_only/len(df_clean)*100:5.2f}%)')
print(f'총 제거:                               {n_drop:>8,}  ({n_drop/len(df_clean)*100:5.2f}%)')
print(f'유효 행:                               {n_keep:>8,}  ({n_keep/len(df_clean)*100:5.2f}%)')

print(f'\n=== 길이 분포 (유효 행만) ===')
print(text_len[~mask_drop].describe().round(0).to_string())

detail_text_clean 빈 값:                     0  ( 0.00%)
detail_text_clean <120자 (빈값 제외):   27,031  (23.87%)
총 제거:                                 27,031  (23.87%)
유효 행:                                 86,200  (76.13%)

=== 길이 분포 (유효 행만) ===
count    86200.0
mean       501.0
std        659.0
min        120.0
25%        172.0
50%        266.0
75%        529.0
max      14654.0


In [8]:
df_clean2 = df_clean[~mask_drop].reset_index(drop=True)
print(f'2차 정제 결과: {len(df_clean2):,} rows × {len(df_clean2.columns)} cols')

df_clean2.to_csv(DST, index=False, encoding='utf-8-sig')
print(f'\nsaved (덮어쓰기): {DST}')
print(f'파일 크기: {DST.stat().st_size / (1024*1024):.1f} MB')

print('\n=== 1차 → 2차 비교 ===')
print(f'1차 통과 : {len(df_clean):>8,} rows')
print(f'2차 통과 : {len(df_clean2):>8,} rows')
print(f'2차 제거 : {len(df_clean) - len(df_clean2):>8,} rows  ({(len(df_clean)-len(df_clean2))/len(df_clean)*100:.2f}%)')
print(f'원본 대비: {len(df_clean2)/len(df)*100:.2f}% 잔존 ({len(df_clean2):,} / {len(df):,})')

2차 정제 결과: 86,200 rows × 8 cols

saved (덮어쓰기): raw/data/company_jobdescription_preprocessing.csv
파일 크기: 117.4 MB

=== 1차 → 2차 비교 ===
1차 통과 :  113,231 rows
2차 통과 :   86,200 rows
2차 제거 :   27,031 rows  (23.87%)
원본 대비: 36.55% 잔존 (86,200 / 235,850)
